# Phase 3 — Candidate Generation & Blocking Analysis
### Amazon ML Challenge: Multi-Source Entity Resolution

**Goal:** Drastically reduce the $O(N_1 \times (N_2 + N_3))$ search space using a refined 6-block deterministic blocking strategy, indexing exclusively across **Source 2 and Source 3** while preserving **Source 1** as the untouched query anchor.

**Sources:**
- **S1 (Query Anchor — strictly preserved):** `train_source1.tsv` / `test_source1.tsv`
- **S2 (Indexed):** `train_source2.tsv` / `test_source2.tsv`
- **S3 (Indexed):** `train_source3.tsv` / `test_source3.tsv`

**Refined `FINAL_BLOCKS` (6 Deterministic Blocks):**
1. `exact_name`: Exact normalized company name
2. `exact_address`: Exact normalized street/full address
3. `exact_postal`: Exact extracted postal code / PIN
4. `country_first_name`: Country + first token of business name
5. `country_region_first`: Country + regional state/city + first token of name
6. `postal_prefix_first`: 3-digit postal zone prefix + first token of name

*Note: Pruned `first_last_name` and `country_first_last` to prevent combinatorial explosion. Oversized block keys (> 5,000 entities) are capped.*

In [2]:
from __future__ import annotations

import csv
import re
import unicodedata
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Set, Tuple

# Auto-detect project root (Local & Google Colab compatible)
if Path("/content/AMAZON_ML_CHALLENGE-main").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE-main")
elif Path("/content/AMAZON_ML_CHALLENGE").exists():
    ROOT = Path("/content/AMAZON_ML_CHALLENGE")
elif Path("..").resolve().name == "AMAZON_ML_CHALLENGE":
    ROOT = Path("..").resolve()
else:
    ROOT = Path(".").resolve()

DATA = ROOT / "student_resource" / "dataset"
OUTPUT_DIR = ROOT / "data" / "processed" / "phase3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_BLOCK_SIZE = 5000

# Refined 6 blocks
FINAL_BLOCKS = [
    "exact_name",
    "exact_address",
    "exact_postal",
    "country_first_name",
    "country_region_first",
    "postal_prefix_first",
]

print(f"Project Root: {ROOT}")
print(f"Dataset Path: {DATA} (Exists: {DATA.exists()})")
print(f"Phase 3 Output Directory: {OUTPUT_DIR}")
print(f"FINAL_BLOCKS ({len(FINAL_BLOCKS)} blocks): {FINAL_BLOCKS}")

## Step 1: Text Normalization and Token Extractors
Robust extraction of postal codes and region tokens from raw/normalized address strings.

In [4]:
def normalize_text(value: str) -> str:
    """Standardizes text for robust blocking key generation."""
    value = str(value or "").strip().lower()
    value = unicodedata.normalize("NFKC", value)
    value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)
    return re.sub(r"\s+", " ", value).strip()

def postal_tokens(address: str) -> List[str]:
    """Extracts candidate postal code patterns from address string."""
    return re.findall(r"\b[a-z0-9][a-z0-9 -]{2,9}\b", address.lower())

def get_region(raw_address: str) -> str:
    """Extracts regional component using comma separation from raw address."""
    parts = [p.strip() for p in raw_address.split(",") if p.strip()]
    if len(parts) >= 2:
        return parts[-2]
    return ""

## Step 2: Block Key Generator (`FINAL_BLOCKS`)
Generates the keys for the 6 target blocks. Compatible with both raw records and Phase 1 normalized records.

In [6]:
def block_key(row: Dict[str, str], block: str) -> str:
    """Generates a blocking key for a given row and blocking strategy."""
    raw_addr = row.get("business_address", "") or ""
    name = normalize_text(row.get("business_name_normalized") or row.get("business_name") or "")
    address = normalize_text(row.get("business_address_normalized") or raw_addr)
    country = normalize_text(row.get("country_normalized") or row.get("country") or "")

    tokens = name.split()
    first = tokens[0] if tokens else ""

    postal = ""
    for tok in postal_tokens(raw_addr):
        if any(ch.isdigit() for ch in tok):
            postal = tok
            break

    if block == "exact_name":
        return name
    if block == "exact_address":
        return address
    if block == "exact_postal":
        return postal
    if block == "country_first_name":
        return f"{country}|{first}" if (country and first) else ""
    if block == "country_region_first":
        region = normalize_text(get_region(raw_addr))
        return f"{country}|{region}|{first}" if (country and region and first) else ""
    if block == "postal_prefix_first":
        prefix = re.sub(r"[^a-z0-9]", "", postal)[:3] if postal else ""
        return f"{prefix}|{first}" if (prefix and first) else ""

    raise ValueError(f"Unknown block: {block}")

## Step 3: Fast Inverted Indexing (Only S2 and S3)
Builds inverted hash tables **exclusively over Source 2 and Source 3** in a single streaming pass.
Source 1 is **not indexed**; it is preserved as the query anchor.

In [8]:
def build_s2_s3_indexes(
    s2_path: Path,
    s3_path: Path,
    blocks: List[str] = FINAL_BLOCKS,
    max_block_size: int = MAX_BLOCK_SIZE,
    max_rows_per_source: int = None
) -> Dict[str, Dict[str, List[str]]]:
    """
    Builds unified inverted indexes over Source 2 and Source 3 only.
    Returns: dict mapping block_name -> {key: [entity_id, ...]}
    """
    indexes: Dict[str, Dict[str, List[str]]] = {b: defaultdict(list) for b in blocks}
    
    for s_label, s_file in [("Source 2", s2_path), ("Source 3", s3_path)]:
        if not s_file.exists():
            print(f"Warning: {s_file} not found. Skipping.")
            continue
            
        print(f"Indexing {s_label} ({s_file.name})...")
        count = 0
        with s_file.open("r", encoding="utf-8", errors="replace", newline="") as f:
            reader = csv.DictReader(f, delimiter="\t")
            for row in reader:
                count += 1
                eid = row["entity_id"]
                
                for b in blocks:
                    key = block_key(row, b)
                    if key and len(indexes[b][key]) < max_block_size:
                        indexes[b][key].append(eid)
                        
                if max_rows_per_source and count >= max_rows_per_source:
                    break
                    
        print(f"  --> Completed {s_label}: {count:,} records indexed.")
        
    for b in blocks:
        print(f"  Block '{b}': {len(indexes[b]):,} unique keys generated.")
        
    return {b: dict(indexes[b]) for b in blocks}

## Step 4: Streaming Candidate Generation Pipeline
Streams Source 1 records (query anchor), matches them against the S2/S3 inverted indexes, deduplicates candidate IDs, and writes directly to `data/processed/phase3/candidate_pairs.tsv`.

In [10]:
def generate_candidate_pairs(
    s1_path: Path,
    s2_path: Path,
    s3_path: Path,
    output_path: Path,
    blocks: List[str] = FINAL_BLOCKS,
    max_block_size: int = MAX_BLOCK_SIZE,
    max_s2_s3_rows: int = None,
    max_s1_rows: int = None
) -> Dict[str, Any]:
    """
    Indexes S2 & S3, streams S1 as query anchor, and writes candidate_pairs.tsv.
    Output format: source1_entity_id \t candidate_entity_ids
    """
    # 1. Build S2/S3 index
    indexes = build_s2_s3_indexes(s2_path, s3_path, blocks=blocks, max_block_size=max_block_size, max_rows_per_source=max_s2_s3_rows)
    
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    total_pairs = 0
    total_s1 = 0
    matched_s1 = 0
    
    print(f"\nStreaming Source 1 queries to {output_path.name}...")
    with (
        s1_path.open("r", encoding="utf-8", errors="replace", newline="") as f1,
        output_path.open("w", encoding="utf-8", newline="") as out
    ):
        reader = csv.DictReader(f1, delimiter="\t")
        writer = csv.writer(out, delimiter="\t")
        writer.writerow(["source1_entity_id", "candidate_entity_ids"])
        
        for row in reader:
            total_s1 += 1
            s1_id = row["entity_id"]
            candidates: Set[str] = set()
            
            for b in blocks:
                key = block_key(row, b)
                if key and key in indexes[b]:
                    candidates.update(indexes[b][key])
                    
            if candidates:
                cand_str = ",".join(sorted(candidates))
                writer.writerow([s1_id, cand_str])
                total_pairs += len(candidates)
                matched_s1 += 1
                
            if total_s1 % 100_000 == 0:
                print(f"  ...processed {total_s1:,} S1 rows | candidate links={total_pairs:,}")
                
            if max_s1_rows and total_s1 >= max_s1_rows:
                break
                
    stats = {
        "total_s1_processed": total_s1,
        "s1_with_candidates": matched_s1,
        "total_candidate_pairs": total_pairs,
        "avg_candidates_per_entity": total_pairs / max(1, matched_s1)
    }
    
    print(f"\n=== CANDIDATE GENERATION COMPLETED ===")
    print(f"- S1 Rows Processed       : {stats['total_s1_processed']:,}")
    print(f"- S1 with Candidates      : {stats['s1_with_candidates']:,}")
    print(f"- Total Candidate Links   : {stats['total_candidate_pairs']:,}")
    print(f"- Average Candidates / S1 : {stats['avg_candidates_per_entity']:.2f}")
    print(f"- Output File             : {output_path}")
    return stats

## Step 5: Execution Entry Point (Ready to Run)
Execute candidate generation using `FINAL_BLOCKS` over S2 and S3, with S1 preserved as anchor.

In [12]:
# File path definitions
TRAIN_S1 = DATA / "train" / "train_source1.tsv"
TRAIN_S2 = DATA / "train" / "train_source2.tsv"
TRAIN_S3 = DATA / "train" / "train_source3.tsv"

OUTPUT_CANDIDATES = OUTPUT_DIR / "candidate_pairs.tsv"

# Run candidate generation (safely indexes only S2 and S3; preserves S1)
# To run on full dataset: set max_s2_s3_rows=None, max_s1_rows=None
# To run on fast sample: set max_s2_s3_rows=500000, max_s1_rows=100000
print("Candidate generation pipeline configured and ready for execution.")
print(f"Target Output: {OUTPUT_CANDIDATES}")